### Spin Test 
This script projects data to surface space and performs a spin test

#### Import Packages

In [1]:
import numpy as np
import pandas as pd
import netneurotools.stats as stats
from neuromaps.datasets import fetch_atlas
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_schaefer2018
import neuromaps
import nibabel as nib
from neuromaps.parcellate import Parcellater
from neuromaps import transforms
from neuromaps import nulls
from scipy.stats import pearsonr, spearmanr
from nilearn.plotting import plot_surf
from nilearn.surface import load_surf_data
from nilearn import surface, plotting, datasets
from neuromaps import datasets, images, nulls, resampling, stats
import netneurotools.datasets
from netneurotools.datasets import fetch_cammoun2012

#### Set Paths and Variables

In [130]:
"""
cammoun = netneurotools.datasets.fetch_cammoun2012(version='fsaverage')
cammoun = cammoun['scale500']
annot_lh = cammoun[0]
annot_rh = cammoun[1]
print(annot_rh)

cam = fetch_cammoun2012()
info = pd.read_csv(cam['info'], sep=',')
info = info[info['scale']== 'scale500']
labels = info['label'][info['structure']=='cortex']
receptor_df = receptor_df[receptor_df['Parcel'].str.contains('|'.join(labels))]
"""

/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-R_deterministic.annot


In [2]:
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500' # 'Schaefer2018_1000Parcels_17Networks' #  #   # # 
nmaps = ["NMDA", "mGluR5", "GABA"]
maps = ["cest", "NMDA", "mGluR5", "GABA"]
group = "HC"
n_permute = 10

# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas

# Read in data
#receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
means= pd.read_csv(inpath + '/means_std_' + dataset + '_' + atlas + '.csv', sep=',') #subjectnmaps

# Reformat data 
means["Parcel"] = means["Parcel"].str.replace(' NZMean', '', regex=False)
means = means[means['hstatus']== group]
means.set_index('Parcel', inplace = True)
#print(means)
"""
receptor_df = receptor_df.reset_index(drop=True)
receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
print(means)
print(receptor_df)
"""
if atlas == 'Schaefer2018_1000Parcels_17Networks':
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
    receptor_df = receptor_df.reset_index(drop=True)
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Read in surface atlases
    schaefer_cifti_path = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_900Parcels_17Networks_order.dlabel.nii'
    annot_rh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/lh.Schaefer2018_1000Parcels_17Networks_order.annot'
    annot_lh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/rh.Schaefer2018_1000Parcels_17Networks_order.annot'
    # Fetch atlas labels
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer['labels']  # These are the MNI coordinates of the parcels
    labels = [label.decode('utf-8') for label in labels]

elif atlas == 'atl-Cammoun2012_res-500':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    # Import surface data
    cammoun = netneurotools.datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    annot_lh = cammoun[0]
    annot_rh = cammoun[1]
    # Import receptor_df
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_cammoun2012_scale500.csv", sep=',')
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)   
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Import labels and remove subcortex
    cam = fetch_cammoun2012()
    info = pd.read_csv(cam['info'], sep=',')
    info = info[info['scale']== 'scale500']
    #labels = info['label'][info['structure']=='cortex'].values
    hemisphere = info['hemisphere'][info['structure']=='cortex'].values
    labels = info['label'][info['structure']=='cortex'].values + hemisphere
    print(labels.shape)
    # Remove subcortex from other dfs
    receptor_df = receptor_df[receptor_df['Parcel'].str.contains('|'.join(labels))]
    print(receptor_df.shape)
    means = means[means.index.str.contains('|'.join(labels))]
    print(means.shape)

(1000,)
(1000, 4)
(52, 6)


### Functions

In [4]:
def scramble(df, spin_df):
    """
    Reorders the rows of the input dataframe `df` based on the indices provided in `spin_df`.
    Each column of `spin_df` represents a permutation of row indices.
    Args:
    df (pd.DataFrame): The original dataframe with data.
    spin_df (pd.DataFrame): DataFrame where each column contains scrambled row indices (1-based).
    Returns:
    null_dfs (list): List of DataFrames with rows reordered according to scrambled indices.
    """
    null_dfs = []  # List to store the permuted DataFrames
    
    # Iterate over each permutation (column) in spin_df
    for col in spin_df.columns:   
        scrambled_indices = spin_df[col].values
        null_df = df.iloc[scrambled_indices] #.reset_index(drop=True)
        null_df.loc[:, 'Parcel'] = labels
        null_df.set_index('Parcel', inplace = True)
        # Append the permuted DataFrame to the list
        null_dfs.append(null_df)
        #print(null_dfs)
    
    return null_dfs
###################

def calculate_null_corrs(null_dfs, nmaps):
    """
    Calculates correlation values for each null df and stores in output df
    Args:
    null_dfs (list): permutation matrices with new indices
    spin_df (pd.DataFrame): DataFrame where each column contains scrambled row indices (1-based).
    Returns:
    null_dfs (list): List of DataFrames with rows reordered according to scrambled indices.
    """
    # Initialize an empty dataframe to store the results
    columns = ['shift_iteration']
    for col in nmaps:
        columns.append(f'{col}_r')
        columns.append(f'{col}_p')
    
    null_stats_df = pd.DataFrame(columns=columns)
    
    # Loop through each null dataframe (each shift iteration)
    for i, null_df in enumerate(null_dfs):
        result_row = {'permutation': i + 1}
        
        # For each nmaps column, calculate r and p values with respect to CESTavg
        for col in nmaps:
            r_value, p_value = pearsonr(null_df['CEST_avg'], null_df[col])
            result_row[f'{col}_r'] = r_value
            result_row[f'{col}_p'] = p_value
        
        # Convert result_row to a DataFrame and concatenate it
        result_row_df = pd.DataFrame([result_row])
        null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)
    
    return null_stats_df
########################

def calculate_real_correlations(df, nmaps):
    """
    This function calculates pearson correlations between CEST_avg and nmap columns.
    Args:
    df (pd.DataFrame): df with cest and nmap parcelwise data
    nmaps (list): names of nmaps
    Returns: 
    Correlation values and p values for parametric associations.
    """
    real_corrs = {}
    for col in nmaps:
        r_value, _ = pearsonr(df['CEST_avg'], df[col])
        real_corrs[col] = r_value
    return real_corrs
#####################

def calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps):
    """
    This function compares real correlations to the null distribution to calculate empirical p-values
    Args:
    real_corrs (pd.DataFrame): df with r and p values for cest-nmap associations
    null_stats_df (pd.DataFrame): df with r and p value for cest-nmap associations per iteration
    nmaps (list): names of nmaps
    Returns: 
    Empirical p values based on null model
    """    
    p_values = {}
    
    # Loop through each column in nmaps
    for col in nmaps:
        # Extract the null r-values for this nmaps column
        null_r_values = null_stats_df[f'{col}_r']
        
        # Calculate the empirical p-value: proportion of null r-values more extreme than the real r-value
        real_r = real_corrs[col]
        
        # Golia's addition - correct for if r values aren't centered on zero:
        null_mean = null_r_values.mean() # Find mean of r nulls
        # Subtract mean from null and real r values.
        null_r_values = null_r_values - null_mean 
        real_r = real_r - null_mean
        
        # Two-tailed test: count both large positive and negative extremes
        extreme_count = np.sum(np.abs(null_r_values) >= np.abs(real_r))
        
        # Compute the empirical p-value
        p_empirical = extreme_count / len(null_r_values)
        p_values[col] = round(p_empirical, 15)
    
    return p_values

### Generate permutations

In [133]:
#### For some reason, shape = (999,n) for S1000 parcellation. All others have correct number of rows.
"""
# Convert cifti to gifti
schaefer_gifti = images.dlabel_to_gifti(schaefer_cifti_path) 
# Spin indices
spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsLR', density='32k',
                                     n_perm=10, seed=1234, parcellation=schaefer_gifti)
print(spin_index.shape) 
"""

"\n# Convert cifti to gifti\nschaefer_gifti = images.dlabel_to_gifti(schaefer_cifti_path) \n# Spin indices\nspin_index = nulls.vazquez_rodriguez(data=None, atlas='fsLR', density='32k',\n                                     n_perm=10, seed=1234, parcellation=schaefer_gifti)\nprint(spin_index.shape) \n"

In [5]:
# Convert fsavg annotations to gifti
"""
if atlas == 'Schaefer2018_1000Parcels_17Networks':
    schaefer_fsgifti = images.annot_to_gifti((annot_lh_schaefer, annot_rh_schaefer))
    # Spin indices
    spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsaverage', density='164k', 
                                         n_perm=10, seed=1234, parcellation=schaefer_fsgifti)
    
    # Reformat spun indices for permutation test
    spin_df = pd.DataFrame(spin_index)
    spin_df.index = pd.RangeIndex(start=1, stop=len(spin_df) + 1, step=1)
    print(spin_df)

elif atlas == 'atl-Cammoun2012_res-500':
    fsgifti = images.annot_to_gifti((annot_lh, annot_rh))
    # Spin indices
    spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsaverage', density='164k', 
                                         n_perm=10, seed=1234, parcellation=fsgifti)
    
    # Reformat spun indices for permutation test
    spin_df = pd.DataFrame(spin_index)
    spin_df.index = pd.RangeIndex(start=1, stop=len(spin_df) + 1, step=1)
    print(spin_df)

"""
    
fsgifti = images.annot_to_gifti((annot_lh, annot_rh))
# Spin indices
spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsaverage', density='164k', # Fix double check density '164k'
                                     n_perm=100, seed=1234, parcellation=fsgifti)

# Reformat spun indices for permutation test
#spin_df = pd.DataFrame(spin_index)
#spin_df.index = pd.RangeIndex(start=1, stop=len(spin_df) + 1, step=1)
print(spin_df)


NameError: name 'spin_df' is not defined

In [21]:
# Fix! Efficient code from Golia:
# e.g. spin_idx has the shape of 400 x 10000
# your real_data: 400 regions

#null_df.loc[:, 'Parcel'] = labels

test = np.array(receptor_df.iloc[:, 1:])
null = test[spin_index]
null_dfs = list()
for row_index in range(null.shape[1]):  
    spindf = pd.DataFrame(null[:, row_index, :], columns=['NMDA', 'mGluR5', 'GABA'])
    spindf['Parcel'] = labels
    null_dfs.append(spindf)
print(len(null_dfs))


100


In [6]:
# null[:,0,0]

In [125]:
pd.options.mode.chained_assignment = None 

In [17]:
# Generate list of null dfs
null_dfs = scramble(receptor_df, spin_df)
#print(len(null_dfs))
#print(null_dfs)
#null = null_dfs[0]
#print(null_dfs)

In [124]:
pd.options.mode.chained_assignment = 'warn'

In [22]:
# Keep only parcels in slice
means_cest = means[['hstatus', 'CEST_avg']]
#print(means_cest)
merged_dfs = []
#print(means_cest)
for null_df in null_dfs:
    merged_df = means_cest.merge(null_df, on='Parcel', how='inner')
    merged_dfs.append(merged_df)
print(merged_dfs[0])

# Need to figure out how to merge when I don't have R/L flag in 

                         Parcel hstatus  CEST_avg      NMDA    mGluR5  \
0    caudalanteriorcingulate_5R      HC  0.140048  1.024301  0.830245   
1                     cuneus_3R      HC  0.884299 -0.539328  0.194108   
2                     cuneus_5R      HC  1.411665 -0.365496  0.465122   
3                frontalpole_1R      HC -0.475798 -0.347017 -0.743088   
4           isthmuscingulate_1R      HC  0.265007 -0.933083 -0.872364   
5           isthmuscingulate_2R      HC  0.592127  0.520408 -0.690516   
6           isthmuscingulate_3R      HC  0.098472 -0.933083 -0.872364   
7           isthmuscingulate_4R      HC  0.246605 -0.501923 -0.843957   
8                   lingual_11R      HC  1.179280  1.022751  0.607009   
9                   lingual_13R      HC  1.987714  0.566947  0.234099   
10              paracentral_11R      HC -0.106498  0.189736 -0.442503   
11              paracentral_12R      HC -0.348940  1.301583  2.436121   
12               paracentral_2R      HC -0.262970 -

In [171]:
# Calculate stats of null distributions
null_stats_df = calculate_null_corrs(merged_dfs, nmaps)
#print(null_stats_df)

real_corrs = calculate_real_correlations(means, nmaps)
empirical_p_values = calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps)



/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_92978/4012987003.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)


Pearson R values with real data:
 {'NMDA': 0.4755258399370453, 'mGluR5': 0.22589118396115393, 'GABA': 0.6970255086107258} 

Empirical p values based on  10  permutations:
NMDA: 0.0122000000000000007743806
mGluR5: 0.3264000000000000234479103
GABA: 0.0001000000000000000047922


In [172]:
print("Pearson R values with real data:\n",real_corrs,"\n\nEmpirical p values based on 10000 permutations:")
for col, p_val in empirical_p_values.items():
    print(f'{col}: {p_val:.25f}') 

Pearson R values with real data:
 {'NMDA': 0.4755258399370453, 'mGluR5': 0.22589118396115393, 'GABA': 0.6970255086107258} 

Empirical p values based on 10000 permutations:
NMDA: 0.0122000000000000007743806
mGluR5: 0.3264000000000000234479103
GABA: 0.0001000000000000000047922
